# 🖼️ CIFAR-10 Image Classification — ANN vs CNN
## Celebal Excellence Internship | Week 4 Assignment | Data Science Track
**Author:** Adil Islam | **Dataset:** CIFAR-10 (60,000 images, 10 classes)

### Objective
Build and compare image classification models on CIFAR-10 using:
- **ANN (Artificial Neural Network)** — Dense-only baseline & deeper variant
- **CNN (Convolutional Neural Network)** — Simple CNN, deep CNN with BatchNorm & Dropout
- **Training Strategies** — Optimizer comparison, LR scheduling, Data Augmentation
- **Analysis** — Performance metrics, confusion matrices, CNN feature maps, architecture insights

### CIFAR-10 Classes
`airplane | automobile | bird | cat | deer | dog | frog | horse | ship | truck`

### Pipeline
```
CIFAR-10 Data → Normalize → One-Hot Encode
       ↓
  ANN Baseline → ANN Deep (BatchNorm)
  CNN Simple   → CNN Deep (BatchNorm + Dropout)
       ↓
  Data Augmentation → CNN + Augmentation
       ↓
  Training Strategies (optimizers, LR schedule)
       ↓
  Full Performance Comparison + Analysis
```


## 📦 1. Install & Import Libraries

In [ ]:
# All required libraries — TensorFlow includes Keras
!pip install -q tensorflow numpy matplotlib seaborn scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                        ModelCheckpoint, LearningRateScheduler)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import classification_report, confusion_matrix

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

## 📂 2. Load & Inspect CIFAR-10

In [ ]:
# CIFAR-10: 60,000 32×32 RGB images across 10 classes
# 50,000 training | 10,000 test — built into Keras datasets

(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.cifar10.load_data()

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
NUM_CLASSES  = 10

print("=" * 50)
print(f"Training images : {X_train_raw.shape}")   # (50000, 32, 32, 3)
print(f"Training labels : {y_train_raw.shape}")
print(f"Test images     : {X_test_raw.shape}")    # (10000, 32, 32, 3)
print(f"Test labels     : {y_test_raw.shape}")
print(f"Pixel range     : [{X_train_raw.min()}, {X_train_raw.max()}]")
print(f"Number of classes: {NUM_CLASSES}")
print("=" * 50)

### 2.1 Sample Images per Class

In [ ]:
fig, axes = plt.subplots(10, 8, figsize=(14, 18))
for cls_idx in range(NUM_CLASSES):
    # Get 8 random samples per class
    idxs = np.where(y_train_raw.flatten() == cls_idx)[0]
    sample_idxs = np.random.choice(idxs, 8, replace=False)
    for col, img_idx in enumerate(sample_idxs):
        ax = axes[cls_idx, col]
        ax.imshow(X_train_raw[img_idx])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(CLASS_NAMES[cls_idx], fontsize=9,
                          fontweight='bold', rotation=0,
                          labelpad=55, va='center')

plt.suptitle('CIFAR-10 — Sample Images per Class (8 each)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Class Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Training set
unique, counts = np.unique(y_train_raw, return_counts=True)
ax1.bar([CLASS_NAMES[i] for i in unique], counts,
        color=plt.cm.tab10(np.linspace(0, 1, 10)), edgecolor='white')
ax1.set_title('Training Set — Class Distribution', fontweight='bold')
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Test set
unique_t, counts_t = np.unique(y_test_raw, return_counts=True)
ax2.bar([CLASS_NAMES[i] for i in unique_t], counts_t,
        color=plt.cm.tab10(np.linspace(0, 1, 10)), edgecolor='white')
ax2.set_title('Test Set — Class Distribution', fontweight='bold')
ax2.set_xlabel('Class')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)

plt.suptitle('Dataset is Perfectly Balanced — 5000 train / 1000 test per class', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Training: {dict(zip([CLASS_NAMES[i] for i in unique], counts))}")
print(f"Test    : {dict(zip([CLASS_NAMES[i] for i in unique_t], counts_t))}")

## ⚙️ 3. Preprocessing

In [ ]:
# ── Normalize pixel values [0,255] → [0.0, 1.0] ────────────────────
# WHY: Neural networks train much faster and more stably when inputs
# are small floats (near zero) rather than large integers (0-255).

X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32')  / 255.0

# ── One-hot encode labels ────────────────────────────────────────
# WHY: Softmax output has 10 neurons; we compare against one-hot vectors.
# categorical_crossentropy requires one-hot; sparse_categorical_crossentropy
# accepts integer labels — we use one-hot for clarity.

y_train_oh = to_categorical(y_train_raw, NUM_CLASSES)
y_test_oh  = to_categorical(y_test_raw,  NUM_CLASSES)

# ── Flatten for ANN (ANN can't process 3D directly) ─────────────
# CNN input: (32, 32, 3) — spatial structure preserved
# ANN input: (3072,)    — flattened, spatial info lost
X_train_flat = X_train.reshape(len(X_train), -1)   # (50000, 3072)
X_test_flat  = X_test.reshape(len(X_test), -1)     # (10000, 3072)

print(f"CNN input shape : {X_train.shape[1:]}")       # (32, 32, 3)
print(f"ANN input shape : {X_train_flat.shape[1:]}")  # (3072,)
print(f"Label shape     : {y_train_oh.shape}")        # (50000, 10)
print()
print(f"Pixel range after normalization: [{X_train.min():.1f}, {X_train.max():.1f}]")

## 🧠 4. ANN — Artificial Neural Network

### Why ANN Struggles with Images
ANNs flatten the image → lose all spatial structure. A 32×32 image has pixels at positions (0,0) and (31,31) — the spatial relationship between neighbors is completely destroyed by flattening. ANNs have no concept of "the pixel next to this one"; every neuron sees every input independently.

### Architecture: ANN Baseline
`Flatten(3072) → Dense(512, ReLU) → Dropout(0.3) → Dense(256, ReLU) → Dropout(0.3) → Dense(10, Softmax)`


In [ ]:
def build_ann_baseline():
    model = models.Sequential([
        # Input layer — already flattened (3072 features)
        layers.Input(shape=(3072,)),

        # Hidden layer 1: 512 neurons, ReLU activation
        # ReLU: max(0,x) — solves vanishing gradient vs sigmoid/tanh
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),          # Drop 30% of neurons randomly during training → prevents overfitting

        # Hidden layer 2: 256 neurons
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),

        # Output: 10 neurons (one per class), Softmax → probabilities summing to 1
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='ANN_Baseline')

    model.compile(
        optimizer='adam',                    # Adam: adaptive LR, best default choice
        loss='categorical_crossentropy',     # Multi-class cross-entropy loss
        metrics=['accuracy']
    )
    return model

ann_baseline = build_ann_baseline()
ann_baseline.summary()
print(f"\nTotal trainable parameters: {ann_baseline.count_params():,}")

In [ ]:
# ── Training Callbacks ─────────────────────────────────────────
# EarlyStopping: stop if val_accuracy doesn't improve for 'patience' epochs
# ReduceLROnPlateau: halve learning rate if stuck → fine-grained convergence

callbacks_base = [
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1)
]

# ── Train ANN Baseline ────────────────────────────────────────
print("Training ANN Baseline...")
hist_ann_base = ann_baseline.fit(
    X_train_flat, y_train_oh,
    epochs=50,
    batch_size=128,
    validation_split=0.1,   # 10% of training data used for validation
    callbacks=callbacks_base,
    verbose=1
)

# ── Evaluate ─────────────────────────────────────────────────
loss_ab, acc_ab = ann_baseline.evaluate(X_test_flat, y_test_oh, verbose=0)
print(f"\n✅ ANN Baseline — Test Accuracy: {acc_ab:.4f} | Test Loss: {loss_ab:.4f}")

### 4.2 ANN Deep — With Batch Normalization

In [ ]:
def build_ann_deep():
    model = models.Sequential([
        layers.Input(shape=(3072,)),

        # BatchNormalization: normalizes activations within each mini-batch
        # WHY: stabilizes training, allows higher LR, acts as mild regularizer
        layers.Dense(1024, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.35),

        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.25),

        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='ANN_Deep_BatchNorm')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

ann_deep = build_ann_deep()
ann_deep.summary()

In [ ]:
print("Training ANN Deep (BatchNorm)...")
hist_ann_deep = ann_deep.fit(
    X_train_flat, y_train_oh,
    epochs=50,
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks_base,
    verbose=1
)

loss_ad, acc_ad = ann_deep.evaluate(X_test_flat, y_test_oh, verbose=0)
print(f"\n✅ ANN Deep (BatchNorm) — Test Accuracy: {acc_ad:.4f} | Test Loss: {loss_ad:.4f}")

## 🔬 5. CNN — Convolutional Neural Network

### Why CNN Works Better for Images
CNNs preserve spatial structure. A Conv2D filter slides across the image — it sees neighboring pixels together, learning local patterns like edges, corners, textures. Key advantages:
- **Local Receptive Fields**: each filter only looks at a small region (e.g. 3×3)
- **Parameter Sharing**: same filter applied everywhere → far fewer params than Dense
- **Translation Invariance**: a cat in the top-left vs bottom-right activates the same filters
- **Hierarchical Features**: early layers learn edges → mid layers learn shapes → deep layers learn objects

### Architecture: Simple CNN
`Conv(32) → Pool → Conv(64) → Pool → Flatten → Dense(128) → Dense(10)`


In [ ]:
def build_cnn_simple():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        # Conv Block 1: learn 32 different 3×3 filters
        # Each filter creates one feature map — detecting one type of pattern
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),    # Downsample: 32×32 → 16×16, keep strongest activations
        layers.Dropout(0.25),

        # Conv Block 2: 64 filters — deeper, more complex patterns
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),    # 16×16 → 8×8
        layers.Dropout(0.25),

        # Classification head
        layers.Flatten(),             # 8×8×64 = 4096 → 4096
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),          # Higher dropout before final layer
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_Simple')

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_simple = build_cnn_simple()
cnn_simple.summary()
print(f"\nTotal trainable parameters: {cnn_simple.count_params():,}")
print(f"Compare ANN Deep params   : {ann_deep.count_params():,}")

In [ ]:
print("Training CNN Simple...")
hist_cnn_simple = cnn_simple.fit(
    X_train, y_train_oh,      # 4D input (50000, 32, 32, 3) — not flattened
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks_base,
    verbose=1
)

loss_cs, acc_cs = cnn_simple.evaluate(X_test, y_test_oh, verbose=0)
print(f"\n✅ CNN Simple — Test Accuracy: {acc_cs:.4f} | Test Loss: {loss_cs:.4f}")

### 5.2 CNN Deep — BatchNorm + L2 Regularization

In [ ]:
def build_cnn_deep():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        # Block 1: 32 filters × 2 conv layers before pooling
        # Stacking conv layers before pooling → richer features at same resolution
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),    # 32×32 → 16×16
        layers.Dropout(0.2),

        # Block 2: 64 filters
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),    # 16×16 → 8×8
        layers.Dropout(0.3),

        # Block 3: 128 filters — high-level abstract features
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),    # 8×8 → 4×4
        layers.Dropout(0.4),

        # Classification head with L2 regularization on Dense weights
        layers.Flatten(),             # 4×4×128 = 2048
        layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_Deep_BatchNorm')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_deep = build_cnn_deep()
cnn_deep.summary()

In [ ]:
# Add LR scheduler: cosine-like step decay
def lr_schedule(epoch):
    if epoch < 20: return 1e-3
    elif epoch < 35: return 5e-4
    else: return 1e-4

callbacks_deep = [
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1),
    LearningRateScheduler(lr_schedule, verbose=0)
]

print("Training CNN Deep (BatchNorm + L2)...")
hist_cnn_deep = cnn_deep.fit(
    X_train, y_train_oh,
    epochs=60,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks_deep,
    verbose=1
)

loss_cd, acc_cd = cnn_deep.evaluate(X_test, y_test_oh, verbose=0)
print(f"\n✅ CNN Deep — Test Accuracy: {acc_cd:.4f} | Test Loss: {loss_cd:.4f}")

## 🔄 6. Training Strategy — Data Augmentation

**Why Data Augmentation?**
CIFAR-10 has 50k training images — relatively small for deep learning. Augmentation artificially expands the dataset by applying random transforms at training time:
- **Horizontal flip**: a cat facing left = same as one facing right
- **Rotation**: slight tilts shouldn't change the label
- **Width/Height shift**: object not always centered

This forces the model to learn invariant features, heavily reducing overfitting.


In [ ]:
# ── ImageDataGenerator with augmentation ─────────────────────
# Only apply augmentation to TRAINING data — test data stays unchanged!
augmentor = ImageDataGenerator(
    horizontal_flip=True,        # Randomly flip images left-right
    rotation_range=15,           # Random rotation up to ±15 degrees
    width_shift_range=0.1,       # Shift image horizontally up to 10%
    height_shift_range=0.1,      # Shift image vertically up to 10%
    zoom_range=0.1,              # Random zoom up to 10%
    fill_mode='nearest'          # Fill empty pixels with nearest value
)
augmentor.fit(X_train)

# ── Visualize augmented samples ───────────────────────────────
sample_img = X_train[:1]  # Take 1 image
aug_gen    = augmentor.flow(sample_img, batch_size=1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
axes[0, 0].imshow(sample_img[0])
axes[0, 0].set_title('Original', fontsize=8)
axes[0, 0].axis('off')

for i in range(1, 8):
    aug_img = next(aug_gen)[0]
    axes[0, i].imshow(np.clip(aug_img, 0, 1))
    axes[0, i].set_title(f'Aug {i}', fontsize=8)
    axes[0, i].axis('off')

sample_img2 = X_train[5:6]
aug_gen2 = augmentor.flow(sample_img2, batch_size=1)
axes[1, 0].imshow(sample_img2[0])
axes[1, 0].set_title('Original', fontsize=8)
axes[1, 0].axis('off')
for i in range(1, 8):
    aug_img2 = next(aug_gen2)[0]
    axes[1, i].imshow(np.clip(aug_img2, 0, 1))
    axes[1, i].set_title(f'Aug {i}', fontsize=8)
    axes[1, i].axis('off')

plt.suptitle('Data Augmentation — Same Image, Different Transformations', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Build CNN Deep + Augmentation (same architecture, different training) ──
cnn_aug = build_cnn_deep()
cnn_aug._name = 'CNN_Deep_Augmented'

callbacks_aug = [
    EarlyStopping(monitor='val_accuracy', patience=12,
                  restore_best_weights=True, verbose=1),
    LearningRateScheduler(lr_schedule, verbose=0)
]

# Use flow() to generate augmented batches on-the-fly during training
print("Training CNN Deep + Data Augmentation...")
hist_cnn_aug = cnn_aug.fit(
    augmentor.flow(X_train, y_train_oh, batch_size=64),
    steps_per_epoch=len(X_train) // 64,
    epochs=60,
    validation_data=(X_test, y_test_oh),    # No augmentation on test!
    callbacks=callbacks_aug,
    verbose=1
)

loss_ca, acc_ca = cnn_aug.evaluate(X_test, y_test_oh, verbose=0)
print(f"\n✅ CNN Deep + Augmentation — Test Accuracy: {acc_ca:.4f} | Test Loss: {loss_ca:.4f}")

## ⚡ 7. Training Strategy — Optimizer Comparison

**Comparing**: SGD with momentum vs Adam vs RMSprop on the same Simple CNN architecture.

| Optimizer | How It Works | Best For |
|-----------|-------------|----------|
| **SGD + Momentum** | Follows gradient, builds velocity | CNNs, when tuned carefully |
| **Adam** | Adaptive LR per parameter, combines momentum + RMSprop | Default choice, fast convergence |
| **RMSprop** | Divides by running mean of squared gradients | RNNs, noisy gradients |


In [ ]:
optimizer_configs = {
    'SGD_Momentum' : keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    'Adam'         : keras.optimizers.Adam(learning_rate=0.001),
    'RMSprop'      : keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9)
}

opt_results = {}

for opt_name, optimizer in optimizer_configs.items():
    print(f"\nTraining with {opt_name}...")
    model = build_cnn_simple()
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy', metrics=['accuracy'])

    cb = [EarlyStopping(monitor='val_accuracy', patience=7,
                        restore_best_weights=True, verbose=0)]
    h = model.fit(X_train, y_train_oh, epochs=30, batch_size=64,
                  validation_split=0.1, callbacks=cb, verbose=0)

    loss_o, acc_o = model.evaluate(X_test, y_test_oh, verbose=0)
    opt_results[opt_name] = {
        'history': h, 'test_acc': acc_o, 'test_loss': loss_o
    }
    print(f"  → Test Accuracy: {acc_o:.4f}")

print("\n✅ Optimizer comparison complete")

## 📈 8. Training History — Learning Curves

In [ ]:
def plot_history(history, title, ax1, ax2):
    epochs = range(1, len(history.history['accuracy']) + 1)
    ax1.plot(epochs, history.history['accuracy'],     label='Train Acc',  color='steelblue')
    ax1.plot(epochs, history.history['val_accuracy'], label='Val Acc',    color='orange', linestyle='--')
    ax1.set_title(f'{title} — Accuracy', fontweight='bold', fontsize=10)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.legend(fontsize=8); ax1.set_ylim(0, 1)

    ax2.plot(epochs, history.history['loss'],     label='Train Loss', color='steelblue')
    ax2.plot(epochs, history.history['val_loss'], label='Val Loss',   color='orange', linestyle='--')
    ax2.set_title(f'{title} — Loss', fontweight='bold', fontsize=10)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.legend(fontsize=8)

histories = [
    (hist_ann_base,  'ANN Baseline'),
    (hist_ann_deep,  'ANN Deep (BN)'),
    (hist_cnn_simple,'CNN Simple'),
    (hist_cnn_deep,  'CNN Deep (BN)'),
    (hist_cnn_aug,   'CNN + Augment'),
]

fig, axes = plt.subplots(5, 2, figsize=(14, 22))
for i, (hist, title) in enumerate(histories):
    plot_history(hist, title, axes[i, 0], axes[i, 1])

plt.suptitle('Training Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 📊 9. Confusion Matrices & Classification Reports

In [ ]:
def plot_confusion_matrix(model, X_test_input, title, ax):
    y_pred = np.argmax(model.predict(X_test_input, verbose=0), axis=1)
    y_true = y_test_raw.flatten()
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.3)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('Actual', fontsize=8)
    ax.tick_params(axis='both', labelsize=7)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

plot_confusion_matrix(ann_baseline,  X_test_flat, 'ANN Baseline', axes[0])
plot_confusion_matrix(cnn_simple,    X_test,      'CNN Simple',   axes[1])
plot_confusion_matrix(cnn_aug,       X_test,      'CNN + Augment',axes[2])

plt.suptitle('Confusion Matrices — Selected Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Detailed classification report for best model ──────────────
print("\n" + "="*60)
print("  Classification Report — CNN Deep + Augmentation (Best Model)")
print("="*60)
y_pred_best = np.argmax(cnn_aug.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test_raw.flatten(), y_pred_best, target_names=CLASS_NAMES))

## 🏆 10. Full Performance Comparison Dashboard

In [ ]:
# ── Compile all results ──────────────────────────────────────
results = {
    'ANN Baseline'      : acc_ab,
    'ANN Deep (BN)'     : acc_ad,
    'CNN Simple'        : acc_cs,
    'CNN Deep (BN+L2)'  : acc_cd,
    'CNN + Augmentation': acc_ca,
}

# Add optimizer results
for name, res in opt_results.items():
    results[f'CNN_{name}'] = res['test_acc']

models_list = list(results.keys())
accs        = list(results.values())
colors      = ['#e74c3c','#e67e22','#2ecc71','#27ae60','#1abc9c',
               '#3498db','#2980b9','#8e44ad']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
bars = ax1.barh(models_list, accs, color=colors[:len(models_list)], edgecolor='white', height=0.6)
ax1.set_xlabel('Test Accuracy')
ax1.set_title('Model Performance Comparison', fontweight='bold', fontsize=13)
ax1.set_xlim(0, 1.05)
ax1.axvline(x=0.1, color='gray', linestyle='--', linewidth=0.8, label='Random baseline (10%)')
for bar, acc in zip(bars, accs):
    ax1.text(acc + 0.005, bar.get_y() + bar.get_height()/2,
             f'{acc:.3f}', va='center', fontsize=9, fontweight='bold')
ax1.legend(fontsize=9)

# Optimizer comparison
opt_names = list(opt_results.keys())
opt_accs  = [opt_results[k]['test_acc'] for k in opt_names]
ax2.bar(opt_names, opt_accs, color=['#e74c3c','#2ecc71','#3498db'], edgecolor='white')
ax2.set_title('Optimizer Comparison (CNN Simple, same arch)', fontweight='bold', fontsize=12)
ax2.set_ylabel('Test Accuracy')
ax2.set_ylim(0.5, 0.85)
for i, (name, acc) in enumerate(zip(opt_names, opt_accs)):
    ax2.text(i, acc + 0.003, f'{acc:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Ranked Results:")
for rank, (model_name, acc) in enumerate(sorted(results.items(), key=lambda x: -x[1]), 1):
    print(f"  {rank}. {model_name:<25} → {acc:.4f} ({acc*100:.1f}%)")

## 🔍 11. CNN Feature Map Visualization

In [ ]:
# Visualize what the first conv layer 'sees' — the feature maps
# This shows how different filters activate on the same input image

# Pick a sample image
sample = X_test[7:8]   # shape: (1, 32, 32, 3)
sample_label = CLASS_NAMES[y_test_raw[7][0]]

# Build a model that outputs after the first Conv2D layer
layer_outputs = [layer.output for layer in cnn_deep.layers
                 if 'conv2d' in layer.name]
feature_model = keras.Model(inputs=cnn_deep.input, outputs=layer_outputs[0])
first_layer_activation = feature_model.predict(sample, verbose=0)

print(f"Input image class : {sample_label}")
print(f"Feature map shape : {first_layer_activation.shape}")  # (1, 32, 32, 32 filters)

# Plot first 16 filter activations
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
axes = axes.flatten()

# First: show original
axes[0].imshow(sample[0])
axes[0].set_title('Original', fontweight='bold', fontsize=8)
axes[0].axis('off')

# Then: filter activations
for i in range(1, 32):
    axes[i].imshow(first_layer_activation[0, :, :, i-1], cmap='viridis')
    axes[i].set_title(f'Filter {i}', fontsize=7)
    axes[i].axis('off')

plt.suptitle(f'CNN First Conv Layer — Feature Maps for "{sample_label}"\n'
             f'Each panel = 1 filter activation (detecting different edges/textures)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 📋 12. Key Insights & Analysis

### Performance Summary
| Model | Architecture | Test Accuracy | Key Strength |
|-------|-------------|---------------|--------------|
| ANN Baseline | 2 Dense + Dropout | ~48-52% | Fast to train, simple |
| ANN Deep (BN) | 4 Dense + BN | ~52-56% | BatchNorm helps stability |
| CNN Simple | 2 Conv + 2 Pool + Dense | ~68-72% | Spatial structure preserved |
| CNN Deep (BN+L2) | 6 Conv + 3 Pool + BN | ~75-80% | Hierarchical features |
| CNN + Augmentation | Same + augmented data | ~78-83% | Best generalization |

### Why CNN Outperforms ANN on Images (3 Core Reasons)
1. **Parameter Efficiency**: A 3×3 Conv filter has 9×C params shared across the entire image. An equivalent Dense layer connecting every pixel would need 3072×N params — ANN overfits faster.
2. **Spatial Hierarchy**: Layer 1 detects edges → Layer 2 detects shapes → Layer 3 detects object parts. ANNs have no such inductive bias.
3. **Translation Invariance**: MaxPooling + weight sharing means the model recognizes a "car" regardless of position in the image.

### Most Confused Classes
- **cat ↔ dog**: Similar textures, shapes, poses — hardest pair in CIFAR-10
- **automobile ↔ truck**: Similar boxy shapes
- **bird ↔ airplane**: Both have wings, appear in sky backgrounds

### Key Training Observations
- **BatchNormalization** accelerated convergence in both ANN and CNN variants
- **Data Augmentation** gave the biggest accuracy boost (~3-5%) with no architectural change — best ROI technique
- **Adam** converges faster than SGD but SGD with tuned momentum can match or beat it with more epochs
- **EarlyStopping** saved significant compute by halting before overfitting set in


In [ ]:
print("=" * 65)
print("  ✅ CIFAR-10 IMAGE CLASSIFICATION — COMPLETE")
print("=" * 65)
print()
print("  ANN Models : ANN Baseline, ANN Deep (BatchNorm)")
print("  CNN Models : CNN Simple, CNN Deep (BN+L2), CNN + Augmentation")
print("  Strategies : LR Scheduling, Data Augmentation, Optimizer Comparison")
print("  Evaluation : Accuracy, Loss Curves, Confusion Matrix,")
print("               Classification Report, Feature Map Visualization")
print()
print("  Author  : Adil Islam")
print("  Dataset : CIFAR-10 (60,000 images, 10 classes)")
print("  Track   : Celebal Technologies — Data Science Internship (CEI)")
print("=" * 65)